In [1]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

# Load the master metadata
df = pd.read_csv(DATA_DIR / "master_dataset.csv")

print(f"Metadata records: {len(df)}")
print()
print("Columns:")
print(df.columns.tolist())

Metadata records: 36

Columns:
['article_id', 'country', 'newspaper', 'date', 'page', 'title', 'theme', 'sub_theme', 'population', 'author', 'source_url', 'archive', 'ocr_quality']


In [2]:
# Find all article text files
text_files = list(RAW_DIR.rglob("*.txt"))

print(f"Text files found: {len(text_files)}")
print()

for file in sorted(text_files):
    print(file.relative_to(RAW_DIR))

Text files found: 36

australia\AUS01.txt
australia\AUS02.txt
australia\AUS03.txt
australia\AUS04.txt
australia\AUS05.txt
australia\AUS06.txt
australia\AUS07.txt
australia\AUS08.txt
australia\AUS09.txt
australia\AUS10.txt
australia\AUS11.txt
australia\AUS12.txt
australia\AUS13.txt
australia\AUS14.txt
australia\AUS15.txt
australia\AUS16.txt
australia\AUS17.txt
british-malaya\MAL01.txt
british-malaya\MAL02.txt
british-malaya\MAL03.txt
british-malaya\MAL04.txt
british-malaya\MAL05.txt
british-malaya\MAL06.txt
british-malaya\MAL07.txt
british-malaya\MAL08.txt
british-malaya\MAL09.txt
british-malaya\MAL10.txt
india\IND01.txt
india\IND02.txt
india\IND03.txt
india\IND04.txt
india\IND05.txt
india\IND06.txt
india\IND07.txt
india\IND08.txt
india\IND09.txt


In [3]:
# Check that every metadata article ID has a matching text file

metadata_ids = set(df["article_id"].astype(str).str.strip())

file_ids = {
    file.stem
    for file in text_files
}

missing_text = sorted(metadata_ids - file_ids)
extra_text = sorted(file_ids - metadata_ids)

print(f"Metadata IDs: {len(metadata_ids)}")
print(f"Text file IDs: {len(file_ids)}")
print()

print("Missing text files:")
print(missing_text if missing_text else "None")

print()

print("Extra text files:")
print(extra_text if extra_text else "None")

Metadata IDs: 36
Text file IDs: 36

Missing text files:
None

Extra text files:
None


In [4]:
# Read every article and calculate basic text statistics

article_stats = []

for file in sorted(text_files):
    text = file.read_text(encoding="utf-8", errors="replace").strip()
    
    # Remove the title (first line) for a rough article-body word count
    lines = text.splitlines()
    body = "\n".join(lines[1:]).strip() if len(lines) > 1 else text
    
    words = body.split()
    
    article_stats.append({
        "article_id": file.stem,
        "country": file.parent.name,
        "word_count": len(words),
        "character_count": len(body)
    })

stats_df = pd.DataFrame(article_stats)

print(f"Articles analysed: {len(stats_df)}")
print(f"Total words: {stats_df['word_count'].sum():,}")
print(f"Average words/article: {stats_df['word_count'].mean():,.0f}")
print(f"Shortest article: {stats_df['word_count'].min():,} words")
print(f"Longest article: {stats_df['word_count'].max():,} words")

Articles analysed: 36
Total words: 15,608
Average words/article: 434
Shortest article: 49 words
Longest article: 1,161 words


In [5]:
# Display every article's word count

display(
    stats_df.sort_values("word_count")[
        ["article_id", "country", "word_count"]
    ].reset_index(drop=True)
)

,article_id,country,word_count
0,AUS06,australia,49
1,AUS01,australia,62
2,AUS13,australia,73
3,AUS11,australia,131
4,AUS04,australia,140
5,AUS05,australia,151
6,IND02,india,163
7,AUS07,australia,170
8,AUS02,australia,179
9,AUS15,australia,183


In [6]:
# Inspect the three shortest articles

shortest_ids = ["AUS06", "AUS01", "AUS13"]

for article_id in shortest_ids:
    file = next(f for f in text_files if f.stem == article_id)
    text = file.read_text(encoding="utf-8", errors="replace").strip()

    print("=" * 80)
    print(f"{article_id} — {len(text.split())} total words")
    print("=" * 80)
    print(text)
    print()

AUS06 — 49 total words
MONGREL WHITES.CORAKI, Friday.At a meeting of the Coraki auxiliary of the United Aborigines' Mission, Pastor G. W. Moore said that the problem of "mongrel whites" who unlawfully visited the reserve at night was making the work of the auxiliary harder. He hoped that the police would catch the offenders.

AUS01 — 62 total words
"NO ADMISSION OF ABORIGINES."MELBOURNE, Friday.The Minister for Defence, Mr. Street, commenting on suggestions that aborigines should be admitted to the militia forces, said he had no intention of taking any action to enable this to be done. Under military regulations every person would be medically examined before enlistment. No person would be enlisted unless he was substantially of European origin or descent.

AUS13 — 73 total words
ABORIGINES’ RATIONS.Allegation Denied.The Chief Secretary, Mr. Tonking, said yesterday that the statement by the president of the Aborigines’ Progressive Association, Mr. W. Ferguson, that the blacks at many p

In [7]:
# Compare corpus size across countries

country_stats = (
    stats_df.groupby("country")["word_count"]
    .agg(
        articles="count",
        total_words="sum",
        average_words="mean",
        median_words="median"
    )
    .round(1)
    .sort_values("total_words", ascending=False)
)

display(country_stats)

,articles,total_words,average_words,median_words
country,,,,
british-malaya,10,6165,616.5,639.0
australia,17,5416,318.6,183.0
india,9,4027,447.4,307.0


In [8]:
# Article length distribution by country

display(
    stats_df.sort_values(["country", "word_count"])[
        ["article_id", "country", "word_count"]
    ].reset_index(drop=True)
)

,article_id,country,word_count
0,AUS06,australia,49
1,AUS01,australia,62
2,AUS13,australia,73
3,AUS11,australia,131
4,AUS04,australia,140
5,AUS05,australia,151
6,AUS07,australia,170
7,AUS02,australia,179
8,AUS15,australia,183
9,AUS12,australia,217
